# Бейзлайн: детекция еды с YOLO11n

Проекту нужна детекция нескольких продуктов с классификацией каждого объекта. YOLO11n — компактный детектор, который обучается на подготовленной разметке YOLO. Исходные веса предварительно обучены на COCO; для распознавания 417 классов проекта требуется дообучение. [Документация модели](https://docs.ultralytics.com/models/yolo11/).

**Данные:** результат ноутбука 03 (`fpb_myfoodrepo_full`). Используем train для обучения, val для выбора лучшего checkpoint, размеченный test — только для итоговой оценки. Неразмеченный FPB test не участвует. Статистика данных и разбиение описаны в `docs/03_eda.md`.

**Метрики:** mAP50 и mAP50–95 (усреднение AP по классам и порогам IoU), precision и recall для объектов. Test не содержит 4 редких класса; его aggregate mAP не характеризует их качество. Метрики не показывают качество оценки калорийности.

Обучение может занять значительное время и потребовать GPU. YOLO11/Ultralytics распространяются по лицензии AGPL-3.0 либо коммерческой лицензии; перед применением в сервисе проверьте условия.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, time
from datetime import datetime

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'scripts/dataset_pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук из папки репозитория calorie_diary.')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import display

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available(), '| MPS:', torch.backends.mps.is_available())

## 1. Настройки и проверка данных

Запустите ноутбук 02 и 03 до этого этапа либо укажите путь к готовой сборке в `DATASET_DIR`. Список размеченного test обязателен. Перед обучением проверяются файлы изображений и меток, дубликаты и пересечение разбиений. После переноса сборки старые абсолютные пути в `data.yaml` могут устареть: здесь создаётся отдельный временный YAML с актуальными путями, исходная сборка не переписывается.

In [ ]:
DATASET_DIR = Path(os.environ.get('CALORIE_DATASET_DIR', ROOT / 'data/processed/fpb_myfoodrepo_full')).expanduser().resolve()
INITIAL_WEIGHTS = Path(os.environ.get('CALORIE_INITIAL_WEIGHTS', ROOT / 'models/yolo11n_coco.pt')).expanduser().resolve()
# Чтобы оценить уже обученную модель, укажите путь к best.pt; None запускает обучение.
EXISTING_WEIGHTS = os.environ.get('CALORIE_EXISTING_WEIGHTS') or None
EXISTING_WEIGHTS = Path(EXISTING_WEIGHTS).expanduser().resolve() if EXISTING_WEIGHTS else None
EPOCHS = int(os.environ.get('CALORIE_EPOCHS', '1'))
IMAGE_SIZE = int(os.environ.get('CALORIE_IMAGE_SIZE', '640'))
BATCH = int(os.environ.get('CALORIE_BATCH', '8'))
WORKERS = int(os.environ.get('CALORIE_WORKERS', '4'))
SEED = 42
# CUDA при наличии; MPS на Apple Silicon, иначе CPU. При ошибках устройства можно указать 'cpu'.
DEVICE = os.environ.get('CALORIE_DEVICE', '0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR = ROOT / 'data/runs' / ('yolo11n_' + RUN_ID)
EXPORT_DIR = ROOT / 'models' / ('food_yolo11n_' + RUN_ID)

config_path = DATASET_DIR / 'data.yaml'
if not config_path.is_file():
    raise FileNotFoundError(f'Нет {config_path}. Укажите путь к готовому датасету или выполните ноутбуки 02 и 03.')
weights_to_load = EXISTING_WEIGHTS or INITIAL_WEIGHTS
if not weights_to_load.is_file():
    raise FileNotFoundError(f'Нет весов {weights_to_load}; см. инструкцию в models/README.md.')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
names = config.get('names')
if not isinstance(names, (list, dict)) or not names:
    raise ValueError('В data.yaml отсутствует список names.')
# Структура соответствует экспорту проекта: images/<split>/... и labels/<split>/....
split_files = {}
seen = {}
for split in ('train', 'val', 'test'):
    listing = DATASET_DIR / f'{split}.txt'
    if not listing.is_file() or not listing.read_text(encoding='utf-8').strip():
        raise ValueError(f'Нет непустого списка размеченной выборки: {listing}. Сначала выполните переразбиение в ноутбуке 03.')
    if split not in config:
        raise ValueError(f'В data.yaml отсутствует {split}; ожидается размеченное разбиение после ноутбука 03.')
    files = []
    for raw in listing.read_text(encoding='utf-8').splitlines():
        if not raw.strip():
            continue
        image = Path(raw.strip()).expanduser()
        image = (image if image.is_absolute() else listing.parent / image).resolve()
        try:
            relative = image.relative_to(DATASET_DIR)
        except ValueError as exc:
            raise ValueError(f'Путь выходит за пределы датасета: {image}') from exc
        if len(relative.parts) < 3 or relative.parts[0] != 'images':
            raise ValueError(f'Ожидается путь images/<split>/...: {image}')
        label = DATASET_DIR / 'labels' / Path(*relative.parts[1:]).with_suffix('.txt')
        if not image.is_file() or not label.is_file():
            raise FileNotFoundError(f'Нет изображения или YOLO-метки: {image}; {label}')
        if image in seen:
            raise ValueError(f'Изображение повторяется в {seen[image]} и {split}: {image}')
        seen[image] = split
        files.append(image)
    if not files:
        raise ValueError(f'Пустой список изображений: {listing}')
    split_files[split] = files
    config[split] = str(listing)
config['path'] = str(DATASET_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=False)
RUNTIME_YAML = RUN_DIR / 'data.yaml'
RUNTIME_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('Датасет:', DATASET_DIR, '| классов:', len(names), '| устройство:', DEVICE)
for split in ('train', 'val', 'test'):
    print(split, ':', len(split_files[split]), 'изображений')
print('Каталог запуска:', RUN_DIR)

## 2. Обучение

Стартуем с COCO-весов, заменяя голову детектора под классы датасета. Для повторной оценки укажите путь к обученному `best.pt` в `EXISTING_WEIGHTS` или переменной `CALORIE_EXISTING_WEIGHTS`. `best.pt` выбирается по метрикам val, а не test. Каждый запуск сохраняется в отдельной папке и не перезаписывает предыдущий. На CPU начните с небольшого значения `EPOCHS` для проверки, но не представляйте такой запуск как результат полноценного обучения.

In [ ]:
TRAIN_DIR = None
if EXISTING_WEIGHTS is None:
    model = YOLO(str(INITIAL_WEIGHTS))
    train_result = model.train(data=str(RUNTIME_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE,
                               batch=BATCH, workers=WORKERS, device=DEVICE, seed=SEED,
                               deterministic=True, project=str(RUN_DIR), name='train',
                               exist_ok=False, plots=True)
    TRAIN_DIR = Path(train_result.save_dir)
    BEST_WEIGHTS = TRAIN_DIR / 'weights/best.pt'
else:
    BEST_WEIGHTS = EXISTING_WEIGHTS
    print('Повторная оценка готовых весов без обучения.')
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Обучение не сохранило {BEST_WEIGHTS}')
print('Лучший checkpoint:', BEST_WEIGHTS)

## 3. Качество на отложенной выборке

Обученная модель загружается заново из `best.pt`. Оцениваем отдельно val и размеченный test с одинаковым размером изображения; точность и полнота зависят от выбранного библиотекой порога. Итоговый test не используется для подбора модели и гиперпараметров. Для редких классов с малым числом объектов оценки нестабильны.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
def ordered_names(values):
    if isinstance(values, list):
        return values
    ordered = [values.get(i, values.get(str(i))) for i in range(len(values))]
    if any(value is None for value in ordered):
        raise ValueError('Словарь классов должен иметь последовательные ID от 0.')
    return ordered
if ordered_names(best_model.names) != ordered_names(names):
    raise ValueError('Классы весов не совпадают с data.yaml: используйте модель, обученную на этой сборке.')
reports = {}
for split in ('val', 'test'):
    result = best_model.val(data=str(RUNTIME_YAML), split=split, imgsz=IMAGE_SIZE,
                            batch=BATCH, workers=WORKERS, device=DEVICE,
                            project=str(RUN_DIR), name='eval_' + split,
                            exist_ok=False, plots=True)
    reports[split] = result
summary = pd.DataFrame({split: {'precision': float(r.box.mp), 'recall': float(r.box.mr),
                                  'mAP50': float(r.box.map50), 'mAP50-95': float(r.box.map)}
                        for split, r in reports.items()}).T
summary.index.name = 'split'
display(summary.round(4))

### Метрики по классам на test

В таблицу попадают только классы с истинными объектами в test. Обязательно смотрите на число объектов: высокое или низкое AP одного редкого класса не является устойчивой оценкой. Классы без объектов отдельно перечислены ниже.

In [ ]:
def class_name(i):
    return names[i] if isinstance(names, list) else names.get(i, names.get(str(i), str(i)))

test_report = reports['test']
indices = [int(i) for i in test_report.box.ap_class_index]
test_classes = pd.DataFrame({'class_id': indices, 'class_name': [class_name(i) for i in indices],
                             'objects': [int(test_report.nt_per_class[i]) for i in indices],
                             'AP50': test_report.box.ap50, 'AP50-95': test_report.box.ap})
test_classes = test_classes.sort_values(['objects', 'class_id'], ascending=[False, True])
display(test_classes.head(20))
missing = sorted(set(range(len(names))) - set(indices))
print('Нет истинных объектов в test:', len(missing), [class_name(i) for i in missing])

## 4. Примеры предсказаний

Показаны первые четыре размеченных test-изображения и предсказанные рамки. Просмотрите также сохранённые графики и примеры из папки оценки: сходные продукты, мелкие объекты и несколько продуктов на одной тарелке особенно важны для качественной проверки. Отсутствие рамки не доказывает отсутствие продукта.

In [ ]:
test_lines = (DATASET_DIR / 'test.txt').read_text(encoding='utf-8').splitlines()
examples = split_files['test'][:4]
fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 5), squeeze=False)
for ax, path in zip(axes.flat, examples):
    prediction = best_model.predict(source=str(path), imgsz=IMAGE_SIZE, conf=0.25, device=DEVICE, verbose=False)[0]
    ax.imshow(prediction.plot()[:, :, ::-1]); ax.set_title(path.name); ax.axis('off')
plt.tight_layout(); plt.show(); plt.close(fig)

## 5. Сохранение весов, метрик и выводы

`best.pt` и `last.pt` — обученные на данных проекта веса; они сохраняются отдельно для каждого запуска вместе с параметрами и метриками. `models/yolo11n_coco.pt` — только исходные веса. Выводы ниже формируются по фактически полученным результатам; качество распознавания необходимо дополнительно проверить на реальных фотографиях пользователей.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(BEST_WEIGHTS, EXPORT_DIR / 'best.pt')
last = TRAIN_DIR / 'weights/last.pt' if TRAIN_DIR else None
if last and last.is_file(): shutil.copy2(last, EXPORT_DIR / 'last.pt')
summary.to_csv(EXPORT_DIR / 'metrics.csv')
test_classes.to_csv(EXPORT_DIR / 'test_per_class.csv', index=False)
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
metadata = {'dataset': str(DATASET_DIR), 'manifest_sha256': sha256(DATASET_DIR / 'manifest.jsonl') if (DATASET_DIR / 'manifest.jsonl').is_file() else None,
            'source_weights_sha256': sha256(weights_to_load), 'best_weights_sha256': sha256(EXPORT_DIR / 'best.pt'),
            'mode': 'evaluate' if EXISTING_WEIGHTS else 'train',
            'split_sha256': {split: sha256(DATASET_DIR / (split + '.txt')) for split in split_files},
            'train_images': len(split_files['train']),
            'val_images': len(split_files['val']),
            'test_images': len(split_files['test']), 'classes': len(names), 'absent_test_classes': missing,
            'epochs_requested': EPOCHS if TRAIN_DIR else None, 'image_size': IMAGE_SIZE, 'batch': BATCH, 'seed': SEED,
            'device': DEVICE, 'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
            'run_dir': str(RUN_DIR)}
(EXPORT_DIR / 'run.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Веса и результаты:', EXPORT_DIR)
print(f"На test: mAP50 = {summary.loc['test', 'mAP50']:.4f}; mAP50–95 = {summary.loc['test', 'mAP50-95']:.4f}; "
      f"precision = {summary.loc['test', 'precision']:.4f}; recall = {summary.loc['test', 'recall']:.4f}.")
print(f'В test не представлены {len(missing)} классов; их качество по этому test оценить нельзя.')
print('Бейзлайн позволяет сравнивать последующие модели; качество калорийности и поведение на реальных фото требуют отдельной проверки.')

gap = float(summary.loc['val', 'mAP50-95'] - summary.loc['test', 'mAP50-95'])
print(f'Разница mAP50–95 между val и test: {gap:+.4f}. '
      'Различие требует проверки на новой выборке; оно само по себе не доказывает переобучение.')
common = test_classes[test_classes['objects'] >= 10].sort_values('AP50-95').head(5)
if len(common):
    print('Классы с наименьшим AP50–95 среди классов с не менее чем 10 объектами на test:')
    display(common[['class_name', 'objects', 'AP50-95']].round(4))
else:
    print('В test нет классов с 10 и более объектами для отдельного сравнения.')
print('Детектор не оценивает массу порции или калорийность; для дневника нужны отдельные методы и проверка на пользовательских снимках.')
